# Logoloc: treino no Colab

Faster R-CNN na FlickrLogos-32. Prepara o ambiente, o dataset e roda o pipeline (`scripts/01` a `08`) numa GPU do Colab.

**Antes de rodar**: `Ambiente de execução -> Alterar tipo de ambiente de execução -> GPU`.

O código vem direto do GitHub. Na Drive precisa estar o `FlickrLogos-32_dataset_v2.zip`. Ajuste `DRIVE_DIR` na célula abaixo pro caminho da pasta.

**Se a sessão cair ou desconectar**, use `Ambiente de execução → Executar tudo` de novo. Os passos de restauração trazem de volta o que já foi processado/treinado.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/TCC II'
import os
assert os.path.exists(DRIVE_DIR), f'Pasta nao encontrada: {DRIVE_DIR}'
print('OK', os.listdir(DRIVE_DIR))

## 1. Código

In [ ]:
PROJECT_DIR = '/content/tcc'
REPO = 'https://github.com/lucaskluuug/tcc.git'

import shutil
if os.path.isdir(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
!git clone -q $REPO $PROJECT_DIR

%cd $PROJECT_DIR
assert os.path.exists('scripts/01_prepare_dataset.py')
!git log --oneline -1
print('OK')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA disponivel:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nenhuma')

## 2. Restaurar backup da Drive

In [ ]:
import subprocess

def restore(name, dest):
    src = f'{DRIVE_DIR}/backup/{name}/'
    if os.path.isdir(src) and os.listdir(src):
        os.makedirs(dest, exist_ok=True)
        subprocess.run(['rsync', '-a', '--info=progress2', src, dest + '/'], check=True)
        print(f'restaurado: {name} -> {dest}')
    else:
        print(f'nada pra restaurar em {name}')

def backup(name, src):
    dest = f'{DRIVE_DIR}/backup/{name}/'
    os.makedirs(dest, exist_ok=True)
    subprocess.run(['rsync', '-a', '--info=progress2', src + '/', dest], check=True)
    print(f'backup: {src} -> {dest}')

restore('processed', 'data/processed')
restore('cache', 'data/cache')
restore('runs', 'outputs/runs')
restore('results', 'outputs/results')

## 3. Dataset

In [ ]:
DATASET_ZIP = f'{DRIVE_DIR}/FlickrLogos-32_dataset_v2.zip'
assert os.path.exists(DATASET_ZIP), f'Nao achei {DATASET_ZIP}'

RAW_DIR = f'{PROJECT_DIR}/data/raw/FlickrLogos-32'
expected = f'{RAW_DIR}/dataset_v2/FlickrLogos-v2/classes'

if os.path.isdir(expected):
    print('OK, ja extraido em', RAW_DIR)
else:
    os.makedirs(f'{RAW_DIR}/dataset_v2', exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP) as zf:
        zf.extractall(f'{RAW_DIR}/dataset_v2')
    assert os.path.isdir(expected)
    print('OK, extraido em', RAW_DIR)
!python scripts/00_inspect_raw.py data/raw/FlickrLogos-32

Se a árvore acima não mostrar `classes/`, `trainset.relpaths.txt` etc. dentro de `dataset_v2/FlickrLogos-v2/`, confira a estrutura do zip.

## 4. Pipeline de dados

In [ ]:
!python scripts/01_prepare_dataset.py

In [ ]:
backup('processed', 'data/processed')
backup('cache', 'data/cache')

## 5. Estimativa de custo

In [ ]:
!python scripts/03_estimate_cost.py \
  --faster-rcnn configs/faster_rcnn/resnet50.yaml configs/faster_rcnn/resnet101.yaml

## 6. Treino

In [ ]:
run_name = "frcnn_r50_v1"
backup_ckpt = f"{DRIVE_DIR}/backup/runs/{run_name}/model_last.pt"
cmd = ["python", "scripts/04_train_faster_rcnn.py", "--model-config", "configs/faster_rcnn/resnet50.yaml",
       "--run-name", run_name, "--checkpoint-backup-dir", f"{DRIVE_DIR}/backup/runs"]
if os.path.exists(backup_ckpt):
    cmd += ["--resume-from", backup_ckpt]
    print(f"retomando de {backup_ckpt}")
subprocess.run(cmd, check=True)

In [ ]:
run_name = "frcnn_r101_v1"
backup_ckpt = f"{DRIVE_DIR}/backup/runs/{run_name}/model_last.pt"
cmd = ["python", "scripts/04_train_faster_rcnn.py", "--model-config", "configs/faster_rcnn/resnet101.yaml",
       "--run-name", run_name, "--checkpoint-backup-dir", f"{DRIVE_DIR}/backup/runs"]
if os.path.exists(backup_ckpt):
    cmd += ["--resume-from", backup_ckpt]
    print(f"retomando de {backup_ckpt}")
subprocess.run(cmd, check=True)

In [ ]:
backup('runs', 'outputs/runs')

## 7. Avaliação: Fluxo 1 + Fluxo 2 sobre P3

In [ ]:
!python scripts/06_evaluate_faster_rcnn.py \
  --checkpoint outputs/runs/frcnn_r50_v1/model_last.pt \
  --model-config configs/faster_rcnn/resnet50.yaml \
  --run-name frcnn_r50_v1

## 8. Consolidação

In [ ]:
!python scripts/08_consolidate_results.py

In [ ]:
backup('results', 'outputs/results')
print('Feito. Resultados em', DRIVE_DIR + '/backup/')